In [1]:
import os
import torch
import chromadb
import open_clip
import numpy as np
from PIL import Image
from pathlib import Path
from dotenv import load_dotenv


In [2]:
# Load same model and tokenizer used for embedding images
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, _ = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
tokenizer = open_clip.get_tokenizer("ViT-H-14")  # the text tokenizer
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [3]:
# embedding text 
@torch.no_grad()
def generate_text_embedding(text: str) -> np.ndarray:
    text = (text or "").strip()   # keep text as-is just strip outer whitespace

    tokens = tokenizer([text]).to(device)          # OpenCLIP tokenizer
    features = model.encode_text(tokens)           # (1, D)

    # same normalization as image embeddings notebook
    features = features / features.norm(dim=-1, keepdim=True)

    return features.cpu().numpy()[0]       

In [4]:
from dotenv import load_dotenv
load_dotenv()

path = Path("D:\\GP\\ECHO\\data\\data\\video_generation\\embeddings\\chroma_db_scripts_pharaohs")
path.mkdir(parents=True, exist_ok=True)

client = chromadb.PersistentClient(path=path)

collection = client.get_or_create_collection(
    name="pharaohs_scripts",
    metadata={"hnsw:space": "cosine"}
)

In [5]:
import boto3

ACCOUNT_ID   = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY   = os.getenv("R2_ACCESS_KEY")
SECRET_KEY   = os.getenv("R2_SECRET_KEY")
BUCKET_NAME  = os.getenv("R2_BUCKET_NAME")

s3_client = boto3.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)
PREFIX = "data/video_generation/outputs/pharaohs_scripts/"

objects = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=PREFIX)
script_files = [obj['Key'] for obj in objects.get('Contents', []) if obj['Key'].lower().endswith('.txt')]

script_id = 0
for key in script_files:
    obj = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)
    script = obj['Body'].read().decode('utf-8', errors="ignore")

    emb = generate_text_embedding(script)
    
    file_name = key.split('/')[-1]
    pharaoh_name = Path(file_name).stem

    collection.add(
        ids=[str(script_id)],
        embeddings=[emb],
        metadatas=[{"pharaoh_name": pharaoh_name,
                    "path": str(key)}],
    )
    script_id += 1

# initialize chroma_db dir(which images come from cloudflare) then upload the directory to the R2 bucket

In [6]:
count = collection.count()
print("Collection size:", count)

Collection size: 80


In [7]:
result = collection.get(include=["embeddings"], limit=1)

embedding_dim = len(result["embeddings"][0])
print("Embedding size:", embedding_dim)

Embedding size: 1024


In [13]:
results = collection.get(
    where={"pharaoh_name": "Akhenaten"}   
)

print("Found:", len(results["ids"]), "documents")

Found: 1 documents


In [20]:
results = collection.get(
    where={"pharaoh_name": "Akhenaten"}  
)

print("Found:", len(results["ids"]), "documents")

# 2️⃣ Update each document
for i in range(len(results["ids"])):

    old_meta = results["metadatas"][i]
    new_meta = {
        **old_meta,
        "pharaoh_name": "Akhenaton",
        "path": old_meta["path"].replace("Akhenaten", "Akhenaton")
    }

    collection.update(
        ids=[results["ids"][i]],
        metadatas=[new_meta]
    )

print("Update completed ✅")

Found: 1 documents
Update completed ✅


In [21]:
results = collection.get(
    where={"pharaoh_name": "Akhenaton"}   
)

print("Found:", len(results["ids"]), "documents")

Found: 1 documents


In [22]:
results = collection.get(
    where={"pharaoh_name": "Amenirdis"}  
)

print("Found:", len(results["ids"]), "documents")

# 2️⃣ Update each document
for i in range(len(results["ids"])):

    old_meta = results["metadatas"][i]
    new_meta = {
        **old_meta,
        "pharaoh_name": "Amenirdis (Daughter of Kashta)",
        "path": old_meta["path"].replace("Amenirdis", "Amenirdis (Daughter of Kashta)")
    }

    collection.update(
        ids=[results["ids"][i]],
        metadatas=[new_meta]
    )

print("Update completed ✅")

Found: 1 documents
Update completed ✅


In [23]:
results = collection.get(
    where={"pharaoh_name": "Amenirdis (Daughter of Kashta)"}   
)

print("Found:", len(results["ids"]), "documents")

Found: 1 documents


In [24]:
results = collection.get(
    where={"pharaoh_name": "Cleopatara VII Philopator"}  
)

print("Found:", len(results["ids"]), "documents")

# 2️⃣ Update each document
for i in range(len(results["ids"])):

    old_meta = results["metadatas"][i]
    new_meta = {
        **old_meta,
        "pharaoh_name": "Cleopatra VII Philopator",
        "path": old_meta["path"].replace("Cleopatara VII Philopator", "Cleopatra VII Philopator")
    }

    collection.update(
        ids=[results["ids"][i]],
        metadatas=[new_meta]
    )

print("Update completed ✅")

Found: 1 documents
Update completed ✅


In [26]:
results = collection.get(
    where={"pharaoh_name": "Cleopatra VII Philopator"}  
)

print("Found:", len(results["ids"]), "documents")

Found: 1 documents


In [27]:
results = collection.get(
    where={"pharaoh_name": "Menkaure"}  
)

print("Found:", len(results["ids"]), "documents")

# 2️⃣ Update each document
for i in range(len(results["ids"])):

    old_meta = results["metadatas"][i]
    new_meta = {
        **old_meta,
        "pharaoh_name": "Menkuare",
        "path": old_meta["path"].replace("Menkaure", "Menkuare")
    }

    collection.update(
        ids=[results["ids"][i]],
        metadatas=[new_meta]
    )

print("Update completed ✅")

Found: 1 documents
Update completed ✅


In [29]:
results = collection.get(
    where={"pharaoh_name": "Menkuare"}  
)

print("Found:", len(results["ids"]), "documents")

Found: 1 documents
